# L2c: Hexadecimal Numbers and Text Representation

This lecture connects dictionaries, sets, strings, and loops to the numerical representation of ASCII and Unicode characters.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Move between characters and their numbers:__ Convert ASCII and Unicode characters to their integer code points and back, using the `convert(...)` function, the `Int(...)` and `Char(...)` constructors, and the purpose-specific `codepoint(...)` accessor. Explain why the code point that identifies a character and the encoding that stores it as bytes are separate decisions. The `Unicode.julia_chartransform(...)` function applies identifier normalization and can return a different character, so it is not a code-point accessor.
> * __Use collections for lookup operations:__ Represent a character collection with a set when the required operation is membership, and a digit table with a dictionary when the operation is lookup by key. The ASCII table, the hexadecimal digit table, and the character membership test each pair one collection with the one operation it supports well.
> * __Implement a base conversion:__ Convert a nonnegative base-10 integer to hexadecimal digits with the division-and-remainder loop, reading the remainders from last to first, and format the result in the standard `U+XXXX` notation with left-padding to at least four digits. The same loop, with a different base and digit table, produces the digits of any target base.

Let's get started!
___

## Setup, Data, and Prerequisites
The setup file activates the course environment and loads the packages and local functions used in this lecture.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
# Load this meeting's file-relative environment, source code, and imports.
include(joinpath(@__DIR__, "Include.jl"));

This lecture uses [the `DataFrames.jl` package](https://dataframes.juliadata.org/stable/) and [the `PrettyTables.jl` package](https://ronisbr.github.io/PrettyTables.jl/stable/) to display character tables. It also loads [the `Unicode` standard library](https://docs.julialang.org/en/v1/stdlib/Unicode/) and the local [`codepoint_hex(...)` function](src/Compute.jl), which formats a character's code point in uppercase `U+XXXX` notation.

___

## ASCII character representation
ASCII emerged from American standards work in the early 1960s and was first published in 1963, with a major revision following in 1967. It gave computer and telecommunications vendors a shared 7-bit code for English text and control signals.

The 7-bit ASCII character set assigns integer values from 0 through 127 to letters, digits, punctuation, and control characters.

> __Relationship to Unicode:__ [Unicode](https://www.unicode.org/standard/standard.html) preserves the original ASCII assignments as its first 128 code points, so an ASCII character has the same numerical value in both systems.

The `ascii_char_dictionary::Dict{Int64,Char}` dictionary maps each ASCII integer to its corresponding Julia [`Char`](https://docs.julialang.org/en/v1/base/strings/#Core.Char). [The `convert(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.convert) converts each integer to a character. The loop then stores the integer-character pair in the dictionary.

In [ ]:
ascii_char_dictionary = let

    # Build a lookup from every 7-bit ASCII code, 0 through 127, to its Char.
    ascii_char_dictionary = Dict{Int64, Char}();
    ASCII_character_range = range(0,stop=127,step=1) |> collect;

    # Array position i selects an ASCII value; that value becomes both the
    # dictionary key and the numerical code converted to Char.
    for i ∈ eachindex(ASCII_character_range)
        my_ascii_char_index = ASCII_character_range[i];
        c = convert(Char, my_ascii_char_index);
        ascii_char_dictionary[my_ascii_char_index] = c;
    end
    ascii_char_dictionary; # value returned by the let block
end

We convert the dictionary into a two-column [`DataFrame`](https://dataframes.juliadata.org/stable/) and display it with [the `pretty_table(...)` function](https://ronisbr.github.io/PrettyTables.jl/stable/). Each row records one numerical value and its character.

In [ ]:
let
    # Sort the dictionary keys because dictionary iteration does not promise
    # the numerical order expected in an ASCII table.
    ASCII_index_array = keys(ascii_char_dictionary) |> collect |> sort;
    character_table_df = DataFrame();
    for i ∈ eachindex(ASCII_index_array)
        my_ascii_char_index = ASCII_index_array[i];
        c = ascii_char_dictionary[my_ascii_char_index];

        # Each row records an ASCII integer in column i and its Char equivalent.
        row = (
            i = my_ascii_char_index,
            character = c
        );
        push!(character_table_df,row);
    end
    # Display the completed two-column table as the let block's result.
    pretty_table(character_table_df)
end

When we built `ascii_char_dictionary`, we explicitly called [the `convert(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.convert) to turn each integer code into its [`Char`](https://docs.julialang.org/en/v1/base/strings/#Core.Char). 

Constructors can perform the same conversions directly: [the `Int(...)` constructor](https://docs.julialang.org/en/v1/base/numbers/#Core.Int) returns the numerical code point of a character, and the `Char(...)` constructor returns the character at a code point. 

Let's convert the character `'+'` by passing it to `Int` with [the `|>` pipe operator](https://docs.julialang.org/en/v1/manual/functions/#Function-composition-and-piping):

In [1]:
# Pass '+' to Int with the pipe operator; the result is decimal code point 43.
'+' |> Int

43

The direct function call returns the same value:

In [2]:
# Call Int directly to produce the same decimal code point.
Int('+')

43

On the other hand, the `Char` constructor converts in the opposite direction, turning the integer back into its character:

In [ ]:
# Convert decimal code point 43 back to '+'.
Char(43)

### Limitations of ASCII
__ASCII provides only 128 assignments__. This is not enough for general scientific and international text. Thus, ASCII has several limitations:

* It represents basic English letters, digits, punctuation, and control characters, but it does not represent most writing systems.
* It has no native support for accented or non-Latin characters such as Greek, Cyrillic, or Arabic text.
* It omits most mathematical symbols, scientific notation, typographic marks, and modern pictographs.

These limitations led to the development of [Unicode](https://www.unicode.org/standard/standard.html).

___

## Unicode characters and collections
[Unicode](https://www.unicode.org/standard/standard.html) defines a common character set for writing systems, mathematical symbols, technical notation, and pictographs. Its code space contains 1,114,112 possible code points, although many positions remain unassigned.

* Unicode assigns every character an integer code point. An encoding such as [UTF-8](https://en.wikipedia.org/wiki/UTF-8) separately determines how that integer is stored as bytes.
* Code points are conventionally written in hexadecimal because each hexadecimal digit corresponds to four binary bits and produces a compact, reversible representation.
* Julia provides [Unicode input](https://docs.julialang.org/en/v1/manual/unicode-input/) and [Unicode library functions](https://docs.julialang.org/en/v1/stdlib/Unicode/) for working with these characters.

Julia permits many Unicode characters in variable names. Let's assign values to two emoji identifiers and confirm that identifier syntax does not change the numerical operations performed on the stored values.

In [ ]:
🌽 = 16; # enter the corn identifier with \:corn: followed by Tab
🍣 = 4; # enter the sushi identifier with \:sushi: followed by Tab

The variables participate in ordinary arithmetic:

In [ ]:
🌽 + 🍣

Multiplication works the same way:

In [ ]:
🌽 * 🍣

Unicode mathematical operators can also appear in Julia expressions. With `🌽 = 16` and `🍣 = 4`, the comparison $🌽\geq{🍣}$ returns `true`:

In [ ]:
🌽 ≥ 🍣 # enter ≥ with \geq followed by Tab

A set supports direct membership tests. For a character set $\mathbb{C}$, the expression $c\in\mathbb{C}$ asks whether the test character `c` belongs to that set.

> Use the [`Set` collection](https://docs.julialang.org/en/v1/base/collections/#Base.Set) when the main operation is membership. A set stores each distinct value once and does not preserve insertion order.

The `C::Set{Char}` variable holds six letters, one control character, and one emoji, which shows that a `Set{Char}` accepts any `Char` rather than only printable ASCII. A set stores each item once and keeps no order, so the printed result may not match the insertion sequence:

In [ ]:
C = let
    # Collect six letters, one control character, and one emoji as Char values.
    C = Set{Char}();
    push!(C,'A');
    push!(C,'B');
    push!(C,'Q');
    push!(C,'R');
    push!(C,'S');
    push!(C,'T');
    push!(C,2 |> Char); # convert code point 2 to its non-printing Char
    push!(C,'🌽') # emoji are Char values even though they are outside ASCII

    C # value returned by the let block
end

Specify the test character `c`:

In [ ]:
c = '🌽';

Check if $c\in\mathbb{C}$:

In [ ]:
c ∈ C # ∈ => \in then tab

Next, we connect these characters to their numerical code points and hexadecimal notation.

___

## Unicode strings and code points
The built-in [Julia `String` type](https://docs.julialang.org/en/v1/base/strings/) is similar (in some ways) to the traditional text model in languages like [C](https://en.wikipedia.org/wiki/C_(programming_language)), namely, a [`String`](https://docs.julialang.org/en/v1/base/strings/) is an ordered, immutable sequence of characters. It is not an array of `Char`, though. Julia stores a `String` as UTF-8 bytes, so one character can occupy several bytes and string indices count bytes rather than characters. 

For example, if we try to access the third character of the string `"a🍣b"` using `"a🍣b"[3]` you'll get [a `StringIndexError`](https://docs.julialang.org/en/v1/base/base/#Base.StringIndexError), because position 3 lands inside the emoji. That is why we use [the `collect(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.collect-Tuple%7BAny%7D) on a string when we want an actual `Vector{Char}`.

In [19]:
"a🍣b" |> x-> x[1] # the index is counting bytes?

'a': ASCII/Unicode U+0061 (category Ll: Letter, lowercase)

__Hmmm. But wait a minute.__ We saw last week that we __could__ access individual characters in a string using their position, e.g., "1010"[3] works fine because each character is a single byte. Let's dig into this further.

The `technical_label::String` variable contains ASCII characters, a Greek letter, and a degree symbol:

In [7]:
technical_label = "ΔP = 25 kPa at 80 °C";

We convert `technical_label::String` to a `Vector{Char}` using [the `collect(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.collect-Tuple%7BAny%7D). We get a `Vector{Char}` where each element is a character from the original string. 

Some of the characters are marked as ASCII/Unicode, while others are Unicode only, but all have `U+XXXX` values, which we call code points. In `U+XXXX` notation, `U+` is followed by at least four hexadecimal digits. Code points above `U+FFFF`, such as `U+1F363`, use the additional digits they require. The hexadecimal text and the base-10 integer represent the same code point.

In [20]:
# Materialize the string's logical characters as a Vector{Char}, not UTF-8 bytes.
technical_label_characters = technical_label |> collect

20-element Vector{Char}:
 'Δ': Unicode U+0394 (category Lu: Letter, uppercase)
 'P': ASCII/Unicode U+0050 (category Lu: Letter, uppercase)
 ' ': ASCII/Unicode U+0020 (category Zs: Separator, space)
 '=': ASCII/Unicode U+003D (category Sm: Symbol, math)
 ' ': ASCII/Unicode U+0020 (category Zs: Separator, space)
 '2': ASCII/Unicode U+0032 (category Nd: Number, decimal digit)
 '5': ASCII/Unicode U+0035 (category Nd: Number, decimal digit)
 ' ': ASCII/Unicode U+0020 (category Zs: Separator, space)
 'k': ASCII/Unicode U+006B (category Ll: Letter, lowercase)
 'P': ASCII/Unicode U+0050 (category Lu: Letter, uppercase)
 'a': ASCII/Unicode U+0061 (category Ll: Letter, lowercase)
 ' ': ASCII/Unicode U+0020 (category Zs: Separator, space)
 'a': ASCII/Unicode U+0061 (category Ll: Letter, lowercase)
 't': ASCII/Unicode U+0074 (category Ll: Letter, lowercase)
 ' ': ASCII/Unicode U+0020 (category Zs: Separator, space)
 '8': ASCII/Unicode U+0038 (category Nd: Number, decimal digit)
 '0': ASCII/Unicode

### Calculate a code point
We will compute the code point for the Greek lunate epsilon `ϵ` and store the character in `test_unicode_char::Char`. The same procedure applies to any Julia `Char`.

In [ ]:
# Use Greek lunate epsilon as the input to both code-point calculations.
test_unicode_char = 'ϵ'

To get the base 10 index of a character, convert it to an integer. `Int(c)` does exactly that, and [the `codepoint(...)` function](https://docs.julialang.org/en/v1/base/strings/#Base.codepoint) is the self-documenting spelling of the same idea.

> __Not `julia_chartransform`:__ [The `Unicode.julia_chartransform(...)` function](https://docs.julialang.org/en/v1/stdlib/Unicode/#Unicode.julia_chartransform) is sometimes mistaken for a code-point accessor. It is not. It applies the normalization Julia uses when parsing identifiers, so it returns a `Char`, and for some inputs a __different__ `Char`: it maps `µ` (`U+00B5`) to `μ` (`U+03BC`). Converting that result to an `Int` would report `956` for a character whose code point is `181`. It happens to be the identity for `ϵ`, which is exactly what makes the mistake hard to catch.

Convert the character to its base-10 code point:

In [ ]:
# Pass the Char to Int to obtain its Unicode code point as a base-10 integer.
test_char_index = test_unicode_char |> Int

[The `codepoint(...)` function](https://docs.julialang.org/en/v1/base/strings/#Base.codepoint) returns the same value as an unsigned integer and makes the purpose of the conversion explicit:

In [ ]:
# Return the same code point with Julia's purpose-specific accessor.
codepoint(test_unicode_char)

The next step converts the base-10 value to hexadecimal and formats it in standard Unicode notation.
> __Hexadecimal digits__: Hexadecimal numbers use decimal digits $(0,1,\dots,9)$ and six extra symbols; the letters `A`, `B`, `C`, `D`, `E`, and `F`, where hexadecimal `A` = decimal 10, through hexadecimal `F` = decimal 15.

The `hexadecimal_digits_dictionary::Dict{Int,Char}` dictionary maps each value from 0 through 15 to its hexadecimal digit. For example, it stores `13 => 'D'`.

In [ ]:
hexadecimal_digits_dictionary = let

    # Build a lookup from each base-16 value to its printable digit.
    hexadecimal_digits_dictionary = Dict{Int,Char}()
    base = 16;

    # Offset from the code point for '0'; the pipeline converts each resulting
    # numerical code point back to Char. Values 0 through 9 are now complete.
    for i ∈ 0:(base - 1)
        hexadecimal_digits_dictionary[i] = '0' + i |> Char
        # Replace the punctuation candidates for values 10 through 15 with A-F.
        if (i > 9)
            hexadecimal_digits_dictionary[i] = 'A' + (i - 10) |> Char 
        end
    end
    hexadecimal_digits_dictionary # value returned by the let block
end

#### Algorithm
The following algorithm converts a base-10 number into Unicode code-point notation. Unicode writes a code point as `U+` followed by its hexadecimal value, left-padded with zeros to a minimum of four digits. Code points above `U+FFFF` use the five or six digits they require.

The `my_code_point::String` variable holds the finished `U+XXXX` string produced by this algorithm.

__Initialize__: Take a nonnegative integer $x\in\mathbb{Z}_{\geq{0}}$ and the `hexadecimal_digits_dictionary::Dict{Int64, Char}`. Set $q\gets{x}$ and let $R$ be an empty remainder array.

While $q\neq{0}$ __do__:
1.  Record the remainder $r = q \bmod 16$ by appending it to $R$.
2.  Replace $q$ with the quotient $\lfloor q/16 \rfloor$.

The remainders come out least significant digit first, so read $R$ __backwards__, looking each value up in the `hexadecimal_digits_dictionary` to get its hexadecimal digit. Left-pad the result with `0` characters until it is at least four digits long, then prepend `U+`.

We implement these steps explicitly and compare the result against the local `codepoint_hex(...)` function, which provides the shorter library-based implementation.

In [ ]:
my_code_point = let

    # Repeated division stores hexadecimal remainders from least significant
    # digit to most significant digit.
    q = test_char_index; 
    remainder_array = Array{Int64,1}();
    while (q != 0)
        r = rem(q,16)
        q = div(q,16)
        push!(remainder_array,r)
    end

    # Reverse the remainders, look up their digit characters, and concatenate
    # them into the hexadecimal representation.
    my_code_point = "";
    for i ∈ reverse(remainder_array)
        tmp = hexadecimal_digits_dictionary[i];
        my_code_point *= tmp |> Char;
    end

    # Left-pad values below U+1000, then pass the padded text to the anonymous
    # function that adds the standard U+ prefix. This is the let-block result.
    my_code_point = lpad(my_code_point, 4, '0') |> x-> "U+"*x
end

# Compare the explicit algorithm with the local function used after this lecture.
reference_code_point = codepoint_hex(test_unicode_char)
(manual = my_code_point, function_result = reference_code_point)

___

## Summary
This lecture connected characters to integer code points, hexadecimal notation, and UTF-8 encoding while using sets and dictionaries for lookup operations.

> __Key Takeaways:__
>
> * **A code point is not an encoding:** The integer identifying a character and the byte sequence used to store it are separate decisions, which is why the same character can occupy a different number of bytes in different encodings. Julia stores a `String` as UTF-8 bytes, so integer string indices count bytes, and an index that lands inside a multibyte character raises an error rather than returning part of a character.
> * **Hexadecimal provides a compact representation:** Base 16 groups four bits per digit, so it states a binary value compactly and reversibly, which is why code points, colors, and memory addresses are written that way. Unicode notation prefixes the hexadecimal value with `U+` and pads it to at least four digits, and code points above the four-digit range use the five or six digits they require.
> * **Collection choice supports the required operation:** A set answers membership questions and stores each distinct value once without order, while a dictionary returns the value stored at a key, which is how the digit table turns each remainder into its hexadecimal character. Matching the collection to the operation keeps the base-conversion loop focused on the numerical steps.

The `U+XXXX` string you built here is the same notation you will see in any Unicode table, and the division-and-remainder loop that produced it is the general recipe for converting between bases.
___